In [9]:
import sqlite3
import warnings
import os
from pathlib import Path

import pandas as pd
warnings.filterwarnings("ignore")

In [10]:
# Schéma relationnel, création des tables

db_path = "../irve_database.db"

# 1. Chargement du dataframe enrichi (niveau granulaire / détaillé)
df_enriched = pd.read_csv("../data/processed/dataset_irve_pop.csv", low_memory=False)

# Normalisation préventive du nom de la colonne du code INSEE
if (
    "code_commune" in df_enriched.columns
    and "code_insee_commune" not in df_enriched.columns
):
  df_enriched["code_insee_commune"] = df_enriched["code_commune"]

# 2. Connexion à SQLite
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

# 3. Création des 4 tables (Schéma DDL)
cursor.executescript("""
CREATE TABLE IF NOT EXISTS COMMUNES (
    code_insee TEXT PRIMARY KEY,
    nom_commune TEXT,
    code_postal TEXT,
    code_departement TEXT,
    population INTEGER,
    region TEXT
);

CREATE TABLE IF NOT EXISTS OPERATEURS (
    id_operateur INTEGER PRIMARY KEY AUTOINCREMENT,
    nom_amenageur TEXT,
    nom_operateur TEXT
);

CREATE TABLE IF NOT EXISTS STATIONS (
    id_station TEXT PRIMARY KEY,
    code_insee TEXT NOT NULL,
    id_operateur INTEGER NOT NULL,
    nom_station TEXT,
    adresse_station TEXT,
    latitude REAL,
    longitude REAL,
    nbre_pdc INTEGER,
    FOREIGN KEY (code_insee) REFERENCES COMMUNES(code_insee),
    FOREIGN KEY (id_operateur) REFERENCES OPERATEURS(id_operateur)
);

CREATE TABLE IF NOT EXISTS POINTS_DE_CHARGE (
    id_pdc TEXT PRIMARY KEY,
    id_station TEXT NOT NULL,
    puissance_nominale REAL,
    tranche_puissance TEXT,
    prise_type_2 BOOLEAN,
    prise_type_combo_ccs BOOLEAN,
    FOREIGN KEY (id_station) REFERENCES STATIONS(id_station)
);
""")

print("✅ Schéma DDL vérifié/créé avec succès.")

# -------------------------------------------------------------------
# INJECTION TABLE 1 : COMMUNES
# -------------------------------------------------------------------
col_insee = (
    "code_insee_commune"
    if "code_insee_commune" in df_enriched.columns
    else "code_insee"
)
col_nom = "nom_commune" if "nom_commune" in df_enriched.columns else "com_name"
col_cp = (
    "code_postal"
    if "code_postal" in df_enriched.columns
    else "consolidated_code_postal"
)
col_dep = (
    "departement" if "departement" in df_enriched.columns else "code_departement"
)
col_reg = "region" if "region" in df_enriched.columns else None

cols_communes_source = [
    c
    for c in [col_insee, col_nom, col_cp, col_dep, "population", "region"]
    if c in df_enriched.columns
]

df_communes = df_enriched[cols_communes_source].dropna(subset=[col_insee]).copy()

df_communes["code_insee"] = (
    df_communes[col_insee]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
    .str.zfill(5)
)

rename_dict = {
    col_nom: "nom_commune",
    col_cp: "code_postal",
    col_dep: "code_departement",
}
if col_reg and col_reg != "region":
  rename_dict[col_reg] = "region"

df_communes = (
    df_communes.rename(columns=rename_dict)
    .drop(columns=[col_insee], errors="ignore")
    .drop_duplicates(subset=["code_insee"])
)

cols_communes_sql = [
    c
    for c in [
        "code_insee",
        "nom_commune",
        "code_postal",
        "code_departement",
        "population",
        "region",
    ]
    if c in df_communes.columns
]
df_communes[cols_communes_sql].to_sql(
    "COMMUNES", conn, if_exists="append", index=False
)


# -------------------------------------------------------------------
# INJECTION TABLE 2 : OPERATEURS
# -------------------------------------------------------------------
df_operateurs = (
    df_enriched[["nom_amenageur", "nom_operateur"]]
    .drop_duplicates()
    .dropna(how="all")
    .reset_index(drop=True)
)
df_operateurs.to_sql("OPERATEURS", conn, if_exists="append", index=False)

df_operateurs_db = pd.read_sql("SELECT * FROM OPERATEURS", conn)
df_merged = df_enriched.merge(
    df_operateurs_db, on=["nom_amenageur", "nom_operateur"], how="left"
)


# -------------------------------------------------------------------
# INJECTION TABLE 3 : STATIONS
# -------------------------------------------------------------------
lat_col = next(
    (
        c
        for c in [
            "latitude_station",
            "latitude",
            "coordonneesXY_latitude",
        ]
        if c in df_merged.columns
    ),
    None,
)
lon_col = next(
    (
        c
        for c in [
            "longitude_station",
            "longitude",
            "coordonneesXY_longitude",
        ]
        if c in df_merged.columns
    ),
    None,
)
nbre_pdc_col = next(
    (
        c
        for c in [
            "nbre_pdc_reel",
            "nbre_pdc",
            "nbre_points_de_charge",
        ]
        if c in df_merged.columns
    ),
    None,
)

cols_stations_req = [
    "id_station_itinerance",
    "code_insee_commune",
    "id_operateur",
    "nom_station",
    "adresse_station",
    lat_col,
    lon_col,
    nbre_pdc_col,
]
cols_stations_presentes = [
    c for c in cols_stations_req if c is not None and c in df_merged.columns
]

df_stations = (
    df_merged[cols_stations_presentes]
    .dropna(subset=["id_station_itinerance", "code_insee_commune", "id_operateur"])
    .copy()
)

df_stations["code_insee"] = (
    df_stations["code_insee_commune"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
    .str.zfill(5)
)

rename_stations = {
    "id_station_itinerance": "id_station",
}
if lat_col:
  rename_stations[lat_col] = "latitude"
if lon_col:
  rename_stations[lon_col] = "longitude"
if nbre_pdc_col:
  rename_stations[nbre_pdc_col] = "nbre_pdc"

df_stations = df_stations.drop_duplicates(
    subset=["id_station_itinerance"]
).rename(columns=rename_stations)

valid_communes = set(df_communes["code_insee"])
df_stations = df_stations[df_stations["code_insee"].isin(valid_communes)]
df_stations["id_operateur"] = df_stations["id_operateur"].astype(int)

cols_stations_sql = [
    c
    for c in [
        "id_station",
        "code_insee",
        "id_operateur",
        "nom_station",
        "adresse_station",
        "latitude",
        "longitude",
        "nbre_pdc",
    ]
    if c in df_stations.columns
]

df_stations[cols_stations_sql].to_sql(
    "STATIONS", conn, if_exists="append", index=False
)


# -------------------------------------------------------------------
# INJECTION TABLE 4 : POINTS_DE_CHARGE
# -------------------------------------------------------------------
cols_pdc_req = [
    "id_pdc_itinerance",
    "id_station_itinerance",
    "puissance_nominale",
    "tranche_puissance",
    "prise_type_2",
    "prise_type_combo_ccs",
]
cols_pdc_presentes = [c for c in cols_pdc_req if c in df_merged.columns]

df_pdc = (
    df_merged[cols_pdc_presentes]
    .dropna(subset=["id_pdc_itinerance", "id_station_itinerance"])
    .copy()
)

df_pdc = df_pdc.drop_duplicates(subset=["id_pdc_itinerance"]).rename(
    columns={
        "id_pdc_itinerance": "id_pdc",
        "id_station_itinerance": "id_station",
    }
)

valid_stations = set(df_stations["id_station"])
df_pdc = df_pdc[df_pdc["id_station"].isin(valid_stations)]

bool_cols = df_pdc.select_dtypes(include=["bool"]).columns
df_pdc[bool_cols] = df_pdc[bool_cols].astype(int)

df_pdc.to_sql("POINTS_DE_CHARGE", conn, if_exists="append", index=False)

# Validation définitive
conn.commit()

# BILAN ET VÉRIFICATION
print("\n--- BASE DE DONNÉES INJECTÉE AVEC SUCCÈS ---")
for table in ["COMMUNES", "OPERATEURS", "STATIONS", "POINTS_DE_CHARGE"]:
  count = pd.read_sql(f"SELECT COUNT(*) as total FROM {table}", conn)[
      "total"
  ].iloc[0]
  print(f"Table {table:<18} : {count:,} lignes")

conn.close()

✅ Schéma DDL vérifié/créé avec succès.

--- BASE DE DONNÉES INJECTÉE AVEC SUCCÈS ---
Table COMMUNES           : 11,667 lignes
Table OPERATEURS         : 4,512 lignes
Table STATIONS           : 47,367 lignes
Table POINTS_DE_CHARGE   : 164,142 lignes
